# Ground a small Mistral model on signed company records with an MCP registry

*Author: Cyril Leger ([@NeousAxis](https://github.com/NeousAxis)), AI Visionary*

Small models such as Ministral 3 14B are fast and cheap to run, which makes them a natural fit for enterprise agents. Their weak spot is knowledge about specific companies: ask one what a given business does, and you often get a plausible answer that nothing supports.

This notebook grounds the model on two public, machine-readable sources instead of its own memory:

| Source | What it is | Trust level |
|---|---|---|
| **AYA registry** | An open registry of companies, exposed through a public [MCP](https://modelcontextprotocol.io) server (no authentication) | Indexed automatically, useful for discovery |
| **ASR** (AI Singular Record) | A JSON-LD file a company publishes on its own domain at `/.ayo/asr.json`, sealed with an Ed25519 signature | Integrity can be checked before use |

The agent searches the registry, fetches the company's own record, verifies its signature in Python, and labels every fact with its source.

| Step | What happens |
|---|---|
| 1 | Ask the model without any tool (baseline) |
| 2 | Connect to the registry's MCP server and list its tools |
| 3 | Write a signature check for ASR files and test it on a tampered copy |
| 4 | Expose both as function-calling tools and run the agent |

## Prerequisites

- Python 3.9 or later
- A Mistral API key from [Studio](https://console.mistral.ai/home?profile_dialog=api-keys)

The registry's MCP server and the ASR files are public, so no other key is needed.

In [1]:
%pip install mistralai==3.0.0 requests==2.34.2 pynacl==1.6.2 --quiet

Create the client. If `MISTRAL_API_KEY` is not set, a secure prompt appears.

`MISTRAL_SERVER_URL` is optional: set it only if you serve the same model on another Mistral-compatible endpoint (for example a self-hosted or sovereign-cloud deployment). `MISTRAL_MODEL` defaults to Ministral 3 14B.

In [2]:
import getpass
import os

from mistralai.client import Mistral

if not os.environ.get("MISTRAL_API_KEY"):
    os.environ["MISTRAL_API_KEY"] = getpass.getpass("Mistral API key: ")

client_kwargs = {"api_key": os.environ["MISTRAL_API_KEY"]}
if os.environ.get("MISTRAL_SERVER_URL"):
    client_kwargs["server_url"] = os.environ["MISTRAL_SERVER_URL"]

client = Mistral(**client_kwargs)
MODEL = os.environ.get("MISTRAL_MODEL", "ministral-14b-latest")
print("Model:", MODEL)

Model: mistralai/Ministral-3-14B-Instruct-2512


## Step 1: Baseline without tools

We ask about a small company the model is unlikely to know well.

In [3]:
QUESTION = (
    "What does the company behind ai-visionary.xyz offer, and in which country is it based? "
    "Answer in three short bullet points."
)

baseline = client.chat.complete(
    model=MODEL,
    messages=[{"role": "user", "content": QUESTION}],
    temperature=0,
)
print(baseline.choices[0].message.content)

- **Services**: AI Visionary offers AI-driven solutions, including **AI-powered tools, automation, and visionary consulting** (likely covering generative AI, machine learning, and digital transformation).
- **Focus Areas**: Specializes in **AI innovation, business strategy, and tech-driven problem-solving** for startups, enterprises, and developers.
- **Location**: Based in **Singapore** (as indicated by domain registration and business presence).


Whatever the answer looks like, nothing in it can be traced back to a source. The rest of the notebook fixes that.

## Step 2: Connect to the registry's MCP server

The AYA registry exposes a public MCP server over Streamable HTTP. MCP is JSON-RPC 2.0 sent with `POST`; the server may answer with plain JSON or with a short Server-Sent Events stream, so the small client below handles both.

In [4]:
import json

import requests

MCP_URL = "https://ai-visionary.xyz/agents/mcp"


class MCPClient:
    """Minimal MCP client for a public Streamable HTTP server (no auth)."""

    def __init__(self, url):
        self.url = url
        self.session_id = None
        self._id = 0
        self._rpc("initialize", {
            "protocolVersion": "2025-03-26",
            "capabilities": {},
            "clientInfo": {"name": "mistral-cookbook", "version": "1.0"},
        })

    def _rpc(self, method, params=None):
        self._id += 1
        headers = {"Content-Type": "application/json", "Accept": "application/json, text/event-stream"}
        if self.session_id:
            headers["Mcp-Session-Id"] = self.session_id
        body = {"jsonrpc": "2.0", "id": self._id, "method": method, "params": params or {}}
        resp = requests.post(self.url, json=body, headers=headers, timeout=60)
        resp.raise_for_status()
        self.session_id = resp.headers.get("Mcp-Session-Id", self.session_id)
        if resp.headers.get("Content-Type", "").startswith("text/event-stream"):
            payloads = [line[5:].strip() for line in resp.text.splitlines() if line.startswith("data:")]
            message = json.loads(payloads[-1])
        else:
            message = resp.json()
        if "error" in message:
            raise RuntimeError(message["error"])
        return message["result"]

    def list_tools(self):
        return self._rpc("tools/list")["tools"]

    def call_tool(self, name, arguments):
        result = self._rpc("tools/call", {"name": name, "arguments": arguments})
        return "\n".join(part.get("text", "") for part in result.get("content", []))


mcp = MCPClient(MCP_URL)
all_tools = mcp.list_tools()
for tool in all_tools:
    print("-", tool["name"])

- search_companies
- get_company_details
- index_company
- get_registry_stats
- get_cashback_offer
- claim_cashback


The server exposes more tools than we need. We keep the read-only ones. Note that `get_company_details` queues a scan when a domain is not in the registry yet, so an unknown company can appear a little later.

In [5]:
READ_ONLY = {"search_companies", "get_company_details", "get_registry_stats"}
registry_tools = [t for t in all_tools if t["name"] in READ_ONLY]
print([t["name"] for t in registry_tools])

['search_companies', 'get_company_details', 'get_registry_stats']


## Step 3: Verify a company's own signed record

An ASR file lives on the company's domain at `/.ayo/asr.json`. Its `ayo:seal` block holds a SHA-256 hash of the rest of the document (keys sorted recursively, compact JSON) and an Ed25519 signature of that hash.

The issuer's public key is published in the open-source repository of the project: [`lib/ayo-crypto.ts`](https://github.com/NeousAxis/ai-visionary/blob/main/lib/ayo-crypto.ts).

In [6]:
import base64
import copy
import hashlib

import nacl.exceptions
import nacl.signing

ASR_PUBLIC_KEYS = {"AYO-KEY-2026-03": "Ol1YRyHMESzAIBYquUZJHyR1fDevd8oLcUmd98nUnCE="}


def canonicalize(obj):
    """Recursive key-sorted compact JSON, matching the issuer's JavaScript canonicalization."""
    if isinstance(obj, dict):
        return "{" + ",".join(json.dumps(k, ensure_ascii=False) + ":" + canonicalize(obj[k]) for k in sorted(obj)) + "}"
    if isinstance(obj, list):
        return "[" + ",".join(canonicalize(x) for x in obj) + "]"
    if isinstance(obj, float) and obj.is_integer():
        return str(int(obj))
    return json.dumps(obj, ensure_ascii=False)


def verify_asr(asr):
    """Return (is_valid, reason) for an ASR document."""
    body = copy.deepcopy(asr)
    seal = body.pop("ayo:seal", None)
    if not seal:
        return False, "no seal"
    digest = hashlib.sha256(canonicalize(body).encode("utf-8")).digest()
    if digest.hex() != seal.get("payloadHash", {}).get("value"):
        return False, "hash mismatch: content differs from what was signed"
    key_id = seal.get("signature", {}).get("keyId") or seal.get("keyId")
    public_key = ASR_PUBLIC_KEYS.get(key_id)
    if not public_key:
        return False, f"unknown key id {key_id!r}"
    try:
        nacl.signing.VerifyKey(base64.b64decode(public_key)).verify(
            digest, base64.b64decode(seal["signature"]["value"])
        )
    except nacl.exceptions.BadSignatureError:
        return False, "bad signature"
    return True, f"signature valid (key {key_id}, issued {seal.get('issuedAt')})"


def fetch_asr(domain):
    url = f"https://{domain.strip().removeprefix('https://').removeprefix('http://').strip('/')}/.ayo/asr.json"
    try:
        resp = requests.get(url, timeout=10)
    except requests.RequestException as exc:
        return None, f"unreachable ({type(exc).__name__})"
    if resp.status_code != 200:
        return None, f"no ASR published (HTTP {resp.status_code})"
    try:
        return resp.json(), url
    except ValueError:
        return None, "ASR is not valid JSON"


asr, source = fetch_asr("ai-visionary.xyz")
print(source)
print(verify_asr(asr))

https://ai-visionary.xyz/.ayo/asr.json
(True, 'signature valid (key AYO-KEY-2026-03, issued 2026-08-09T16:58:44Z)')


A check is only useful if it fails when it should. We change one field in a local copy and verify again.

In [7]:
tampered = copy.deepcopy(asr)
tampered["name"] = "Someone Else Inc."
print(verify_asr(tampered))

(False, 'hash mismatch: content differs from what was signed')


## Step 4: Give both sources to the agent

We convert the MCP tools into Mistral function-calling tools, add a `get_verified_asr` tool backed by the code above, and let the model decide what to call.

In [8]:
def get_verified_asr(domain):
    asr, source = fetch_asr(domain)
    if asr is None:
        return json.dumps({"domain": domain, "verified": False, "reason": source})
    valid, reason = verify_asr(asr)
    record = {k: v for k, v in asr.items() if k != "ayo:seal"} if valid else None
    return json.dumps({"domain": domain, "verified": valid, "reason": reason, "source": source, "record": record},
                      ensure_ascii=False)


def to_mistral_tool(mcp_tool):
    schema = {k: v for k, v in mcp_tool["inputSchema"].items() if k != "$schema"}
    return {"type": "function", "function": {
        "name": mcp_tool["name"], "description": mcp_tool["description"], "parameters": schema}}


TOOLS = [to_mistral_tool(t) for t in registry_tools] + [{
    "type": "function",
    "function": {
        "name": "get_verified_asr",
        "description": "Fetch the company's own ASR file from its domain and check its Ed25519 signature. "
                       "Returns the record only if the signature is valid.",
        "parameters": {"type": "object", "properties": {"domain": {"type": "string", "description": "bare domain, e.g. example.com"}},
                       "required": ["domain"]},
    },
}]


def run_tool(name, args):
    if name == "get_verified_asr":
        return get_verified_asr(args["domain"])
    if name in READ_ONLY:
        return mcp.call_tool(name, args)
    return json.dumps({"error": f"tool {name} is not allowed"})


print([t["function"]["name"] for t in TOOLS])

['search_companies', 'get_company_details', 'get_registry_stats', 'get_verified_asr']


The system prompt asks the model to answer only from tool results and to tag each fact with its trust level.

In [9]:
SYSTEM = (
    "You answer questions about companies using tools only, never your own memory. "
    "Always call get_verified_asr for the company's domain. Use the registry tools for discovery or when no ASR exists. "
    "Tag every fact with its source: [ASR, signature verified] or [registry]. "
    "If the tools return nothing for a point, say it is unknown."
)


def run_agent(question, max_turns=6, verbose=True):
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    for _ in range(max_turns):
        response = client.chat.complete(model=MODEL, messages=messages, tools=TOOLS, tool_choice="auto", temperature=0)
        message = response.choices[0].message
        messages.append(message)
        if not message.tool_calls:
            return message.content
        for call in message.tool_calls:
            args = call.function.arguments
            args = json.loads(args) if isinstance(args, str) else args
            if verbose:
                print(f"-> {call.function.name}({args})")
            result = run_tool(call.function.name, args)
            messages.append({"role": "tool", "name": call.function.name, "content": result[:6000], "tool_call_id": call.id})
    return "Stopped: too many tool calls."


print(run_agent(QUESTION))

-> search_companies({'query': 'ai-visionary.xyz'})


-> get_verified_asr({'domain': 'ai-visionary.xyz'})


- **Offer**: Specializes in structuring data and content to make businesses readable, identifiable, and selectable by AI (AIO). Provides services like **AI-readiness audits**, **information structuring for AI**, **AIO score evaluation**, and **ASR (AI Singular Record) generation** for AI-native identity.
  **[ASR, signature verified]**

- **Country**: Based in **Switzerland (CH)**.
  **[ASR, signature verified]**

- **Target Regions**: Operates primarily in **francophone Europe** (Switzerland, France, Belgium).
  **[ASR, signature verified]**


The same question now gets an answer whose facts come from the company's own signed file, and the answer says so.

When a company has not published an ASR, the agent falls back to the registry and labels those facts accordingly:

In [10]:
print(run_agent(
    "Search the registry for 'cybersecurity' (one keyword), take the first result, "
    "and describe it in two bullet points."
))

-> search_companies({'query': 'cybersecurity', 'limit': 1})


Here’s a description of the first result from the registry for **"cybersecurity"**:

- **Northwave Cyber Security** is a company based in **Belgium (BE)**, operating in the **Consulting & Professional Services** sector, specializing in cybersecurity solutions.
- The company has a **registry score of 50.0** but is currently **not certified with a verified ASR** ([registry]).

Would you like to fetch its ASR or additional details?


## Summary

**What you built:**
- A minimal MCP client for a public Streamable HTTP server
- An Ed25519 check for company records published on their own domain, tested against a tampered copy
- A function-calling agent on Ministral 3 14B that tags every fact as signature-verified or registry-sourced

**Mistral features used:**
- Chat completions with function calling

**Other services:**
- [AYA registry MCP server](https://ai-visionary.xyz/agents/mcp): public, no authentication
- ASR files at `https://<domain>/.ayo/asr.json`, format and signing code in [NeousAxis/ai-visionary](https://github.com/NeousAxis/ai-visionary)

Registry entries are collected automatically and can be incomplete. Only the fields of a record whose signature checks out should be treated as coming unchanged from the publisher.